# 🎬 VIDEO-FREE · 5. Director: el video musical completo
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ivanherrerabaquero-dev/VIDEO-FREE/blob/main/colab/5_director_video.ipynb)

Este cuaderno hace **todo el video** a partir de tu canción y tus personajes:

1. **Plano de tomas.** Escribe la historia sobre los compases de la canción: dron de apertura, él se sube al Camaro, primeros planos cantando, la copiloto tarareando, manos en el volante, guitarra, recuerdo en blanco y negro, noche, fantasma y cierre.
2. **Fotogramas clave fotorrealistas** (RealVisXL). La cara de tu personaje se conserva **píxel por píxel** desde su recorte; la IA pinta el auto, la carretera y la luz alrededor. Si entrenaste tu personaje con el cuaderno 6, también puede crearlo en cualquier pose.
3. **Movimiento real** (Wan 2.2 imagen → video). Cada fotograma cobra vida: camina, abre la puerta, se sube, maneja, el pelo se mueve con el viento, la cámara vuela.
4. **Labios** (LatentSync). En las tomas donde canta, la boca sigue la voz de la canción.
5. **Montaje.** Cortes al compás, color de cine según la hora (dorado, atardecer, noche, blanco y negro), grano, franjas de cine y la canción.

| GPU | Video | Tiempo para 30 s de canción |
|---|---|---|
| A100 (Colab Pro) | Wan 2.2 5B, 1280x704 | ~40–60 min |
| L4 (Colab Pro) | Wan 2.2 5B, 1280x704 | ~1,5–2,5 h |
| T4 gratis | Wan 2.2 5B comprimido (GGUF), 832x480, **experimental** | ~2–3 h |

- **Puedes cortar y seguir.** Cada fotograma y cada toma se guardan en tu Drive (`VIDEO-FREE/director/`) apenas quedan listos. Si Colab se desconecta, vuelve a ejecutar las celdas: solo hace lo que falta.
- **Licencias:** RealVisXL (OpenRAIL++), Wan 2.2 (Apache 2.0), LatentSync (Apache 2.0 / OpenRAIL++). Detalle en `colab/LICENCIAS-TERCEROS.md`.

**Antes de empezar (una sola vez):**
1. *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU.*
2. Crea un token de GitHub (*Settings → Developer settings → Fine-grained tokens*) con acceso solo a **VIDEO-FREE** y permiso **Contents: Read and write**.
3. En Colab, abre el icono de **llave (Secretos)**, crea `GITHUB_TOKEN`, pega el token y activa *Acceso al cuaderno*.



In [ ]:
#@title 1 · Preparar Colab y conectar con tu repo
REPO = "ivanherrerabaquero-dev/VIDEO-FREE"  #@param {type:"string"}
RAMA = "main"  #@param {type:"string"}
#@markdown Guarda copias en tu Google Drive (Colab te pedirá permiso para acceder a Drive).
USAR_DRIVE = True  #@param {type:"boolean"}
import os, sys, subprocess
if os.path.exists("/content/VIDEO-FREE/.git"):
    subprocess.run(["git", "-C", "/content/VIDEO-FREE", "pull", "-q"], check=False)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", RAMA, f"https://github.com/{REPO}", "/content/VIDEO-FREE"], check=True)
sys.path.insert(0, "/content/VIDEO-FREE/colab")
import importlib, comun; importlib.reload(comun)
info = comun.preparar(REPO, RAMA, usar_drive=USAR_DRIVE)
VRAM = info["gpu"].get("vram_gb", 0)

In [ ]:
#@title 2 · Instalar (2–3 min)
!pip install -q -U "diffusers>=0.35" transformers accelerate peft gguf ftfy sentencepiece imageio imageio-ffmpeg
import importlib, director; importlib.reload(director)
print("Listo")

In [ ]:
#@title 3 · El plano de tomas (la historia sobre los compases)
PROYECTO = "carretera-sin-ti.json"  #@param {type:"string"}
FORMATO = "horizontal"  #@param ["horizontal", "vertical"]
RITMO = "auto"  #@param ["auto", "rapido", "pausado"]
#@markdown «rapido» corta cada compás, como un video urbano; «auto» lo hace si la canción dura menos de un minuto.
SEMILLA = 11  #@param {type:"integer"}
USAR_PLANO_GUARDADO = True  #@param {type:"boolean"}
#@markdown Si ya hiciste un plano antes para este proyecto, se usa ese (con tus cambios).
import shutil
from pathlib import Path
p = comun.proyecto(PROYECTO)
duracion = p.get("duracion") or comun.duracion(comun.bajar(p["audio"]))
guardado = (p.get("director") or {}).get("plano")
if USAR_PLANO_GUARDADO and guardado and (p.get("director") or {}).get("formato") == FORMATO:
    plano = guardado
    print("Uso el plano guardado en el proyecto.")
else:
    plano = director.planear(duracion, p.get("bpm") or 95, p.get("letra"), semilla=SEMILLA, ritmo=RITMO)
nombre = comun.slug(p.get("titulo", "video"))
CARPETA = f"director/{nombre}_{FORMATO}"  # en tu Drive: VIDEO-FREE/director/...
print(director.tabla(plano, p.get("letra")))
print(f"\n{len(plano)} tomas · {sum(t['canta'] for t in plano)} con labios sincronizados · {duracion:.1f} s")

def en_carpeta(archivo):
    return comun.en_drive(f"{CARPETA}/{archivo}")

def guardar_en_carpeta(local):
    comun.a_drive(local, f"{CARPETA}/{Path(local).name}")

In [ ]:
#@title 3b · (Opcional) Cambiar una toma
TOMA = 0  #@param {type:"integer"}
#@markdown Número de la toma en la tabla (0 = no cambiar nada). Deja vacío lo que no quieras cambiar. Al cambiarla, se vuelve a hacer desde cero.
NUEVA_IMAGEN = ""  #@param {type:"string"}
#@markdown Qué se ve, en inglés (ej. `close-up of a young man singing in the rain inside a black 1969 convertible`).
NUEVO_MOVIMIENTO = ""  #@param {type:"string"}
#@markdown Qué pasa durante la toma, en inglés (ej. `he slams the steering wheel and shouts the lyrics`).
NUEVO_PERSONAJE = ""  #@param {type:"string"}
#@markdown Archivo de `medios/personajes/` (ej. `actor_canta.png`) o `ninguno`.
LABIOS = "igual"  #@param ["igual", "sí", "no"]
NUEVA_SEMILLA = 0  #@param {type:"integer"}
if TOMA:
    t = plano[TOMA - 1]
    if NUEVA_IMAGEN: t["imagen"] = f"{NUEVA_IMAGEN}, {director.ESTILO_FOTO}"
    if NUEVO_MOVIMIENTO: t["mueve"] = f"{NUEVO_MOVIMIENTO}, {director.MOVIMIENTO_COMUN}"
    if NUEVO_PERSONAJE:
        t["personaje"] = None if NUEVO_PERSONAJE == "ninguno" else NUEVO_PERSONAJE
        t["encuadre"] = t.get("encuadre") or "primer_plano"
    if LABIOS != "igual": t["canta"] = LABIOS == "sí"
    if NUEVA_SEMILLA: t["semilla"] = NUEVA_SEMILLA
    for f in (f"clave_{TOMA:02d}.png", f"toma_{TOMA:02d}.mp4", f"labios_{TOMA:02d}.mp4"):
        viejo = en_carpeta(f)
        if viejo: viejo.unlink()
    print(director.tabla([t], p.get("letra")))

In [ ]:
#@title 4 · Fotogramas clave fotorrealistas (RealVisXL)
MODO_PERSONAJE = "recorte (cara exacta)"  #@param ["recorte (cara exacta)", "entrenado (pose libre)"]
#@markdown «entrenado» usa el personaje que entrenaste con el cuaderno 6; si no hay entrenamiento, usa el recorte.
ARMONIZAR = 0.15  #@param {type:"slider", min:0, max:0.4, step:0.05}
#@markdown Repasa toda la imagen para que la luz del personaje coincida con la del lugar. Más alto = luz más natural, pero la cara cambia un poco.
PASOS_IMAGEN = 30  #@param {type:"slider", min:20, max:50, step:5}
REHACER = ""  #@param {type:"string"}
#@markdown Números de toma a repetir, separados por comas (ej. `3,5`). Vacío = solo las que faltan.
import gc, json, torch
from PIL import Image
from IPython.display import display
from diffusers import (StableDiffusionXLPipeline, StableDiffusionXLInpaintPipeline, StableDiffusionXLImg2ImgPipeline,
                       DPMSolverMultistepScheduler)
rehacer = {int(x) for x in REHACER.replace(" ", "").split(",") if x}
W, H = (1344, 768) if FORMATO == "horizontal" else (768, 1344)
por_n = {t["n"]: t for t in plano}

# Personajes entrenados con el cuaderno 6 (medios/lora/<nombre>.json + el archivo en Drive o en el repo)
loras = {}
for quien in ("actor", "actriz"):
    datos = comun.leer(f"medios/lora/{quien}.json")
    if not datos:
        continue
    archivo = comun.en_drive(f"lora/{quien}.safetensors")
    if not archivo:
        try:
            archivo = comun.bajar(f"medios/lora/{quien}.safetensors")
        except FileNotFoundError:
            archivo = None
    if archivo:
        loras[quien] = (Path(archivo), json.loads(datos)["palabra"])
usar_lora = MODO_PERSONAJE.startswith("entrenado") and bool(loras)
print("Personajes entrenados:", ", ".join(f"{k} ({v[1]})" for k, v in loras.items()) or "ninguno")

claves, pendientes = {}, []
for t in plano:
    f = comun.TRABAJO / f"clave_{t['n']:02d}.png"
    previo = en_carpeta(f.name)
    if previo and t["n"] not in rehacer:
        shutil.copy(previo, f); claves[t["n"]] = f
    else:
        pendientes.append(t)
print(f"{len(claves)} ya hechos (en Drive) · {len(pendientes)} por pintar")

def de_quien(t):
    return "actriz" if (t.get("personaje") or "").startswith("actriz") else "actor"

con_recorte = [t for t in pendientes if t.get("personaje") and not (usar_lora and de_quien(t) in loras)]
con_texto = [t for t in pendientes if t not in con_recorte]

def planificador(pipe):
    return DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, use_karras_sigmas=True, algorithm_type="sde-dpmsolver++")

def guardar_clave(n, img):
    f = comun.TRABAJO / f"clave_{n:02d}.png"
    img.save(f); guardar_en_carpeta(f); claves[n] = f

# Fase A: el personaje con su recorte; la IA pinta auto, carretera y luz alrededor
pintadas = {}
if con_recorte:
    pipe = StableDiffusionXLInpaintPipeline.from_pretrained("OzzyGT/RealVisXL_V4.0_inpainting", torch_dtype=torch.float16,
                                                            variant="fp16").to("cuda")
    pipe.scheduler = planificador(pipe)
    for t in con_recorte:
        recorte = Image.open(comun.bajar(f"medios/personajes/{t['personaje']}"))
        lienzo, mascara = director.componer_recorte(recorte, W, H, t.get("encuadre") or "primer_plano")
        g = torch.Generator("cuda").manual_seed(t["semilla"])
        pintado = pipe(prompt=t["imagen"], negative_prompt=director.NEGATIVO_FOTO, image=lienzo, mask_image=mascara,
                       width=W, height=H, strength=0.99, num_inference_steps=PASOS_IMAGEN, guidance_scale=6.0,
                       generator=g).images[0].resize((W, H))
        # la cara y el cuerpo originales vuelven encima, tal cual
        pintadas[t["n"]] = Image.composite(pintado, lienzo, mascara)
        print("Pintada la toma", t["n"])
    del pipe; gc.collect(); torch.cuda.empty_cache()

# Fase B: tomas sin personaje (o con el personaje entrenado) y armonizar la luz de las de la fase A
if con_texto or (pintadas and ARMONIZAR > 0):
    pipe = StableDiffusionXLPipeline.from_pretrained("SG161222/RealVisXL_V5.0", torch_dtype=torch.float16, variant="fp16").to("cuda")
    pipe.scheduler = planificador(pipe)
    if usar_lora:
        for quien, (archivo, _) in loras.items():
            pipe.load_lora_weights(str(archivo.parent), weight_name=archivo.name, adapter_name=quien)
    for t in con_texto:
        prompt = t["imagen"]
        if usar_lora and t.get("personaje") and de_quien(t) in loras:
            palabra = loras[de_quien(t)][1]
            prompt = prompt.replace("young man", palabra).replace("young woman", palabra)
            pipe.enable_lora(); pipe.set_adapters([de_quien(t)], adapter_weights=[0.9])
        elif usar_lora:
            pipe.disable_lora()
        g = torch.Generator("cuda").manual_seed(t["semilla"])
        img = pipe(prompt=prompt, negative_prompt=director.NEGATIVO_FOTO, width=W, height=H,
                   num_inference_steps=PASOS_IMAGEN, guidance_scale=5.0, generator=g).images[0]
        guardar_clave(t["n"], img); print("Pintada la toma", t["n"])
    if pintadas and ARMONIZAR > 0:
        if usar_lora:
            pipe.disable_lora()
        repaso = StableDiffusionXLImg2ImgPipeline(**pipe.components)
        for n, img in list(pintadas.items()):
            g = torch.Generator("cuda").manual_seed(por_n[n]["semilla"])
            pintadas[n] = repaso(prompt=por_n[n]["imagen"], negative_prompt=director.NEGATIVO_FOTO, image=img, strength=ARMONIZAR,
                                 num_inference_steps=PASOS_IMAGEN, guidance_scale=5.0, generator=g).images[0].resize((W, H))
        del repaso
    del pipe; gc.collect(); torch.cuda.empty_cache()
for n, img in pintadas.items():
    guardar_clave(n, img)

hoja = director.hoja_de_contactos([claves[t["n"]] for t in plano if t["n"] in claves], comun.TRABAJO / "fotogramas.jpg",
                                  columnas=4 if FORMATO == "horizontal" else 6, ancho=420 if FORMATO == "horizontal" else 240)
display(Image.open(hoja))
print("¿Alguno no te gusta? Cambia la SEMILLA de esa toma en la celda 3b, o escribe su número en REHACER, y vuelve a ejecutar esta celda.")

In [ ]:
#@title 5 · Dar vida a cada toma (Wan 2.2 · imagen → video)
PASOS = 0  #@param {type:"slider", min:0, max:50, step:5}
#@markdown 0 = automático (30 en A100/L4, 20 en T4). Más pasos = más detalle y más tiempo.
REHACER_VIDEO = ""  #@param {type:"string"}
import gc, numpy as np, torch
from diffusers import AutoencoderKLWan, WanPipeline, WanImageToVideoPipeline, WanTransformer3DModel, UniPCMultistepScheduler
from diffusers.utils import export_to_video
M5 = "Wan-AI/Wan2.2-TI2V-5B-Diffusers"
GGUF = "https://huggingface.co/QuantStack/Wan2.2-TI2V-5B-GGUF/blob/main/Wan2.2-TI2V-5B-Q8_0.gguf"
GRANDE = VRAM >= 22
dtype = torch.bfloat16 if GRANDE else torch.float16
if GRANDE:
    aw, ah = (1280, 704) if FORMATO == "horizontal" else (704, 1280)
else:
    aw, ah = (832, 480) if FORMATO == "horizontal" else (480, 832)
pasos = PASOS or (30 if GRANDE else 20)
rehacer_v = {int(x) for x in REHACER_VIDEO.replace(" ", "").split(",") if x}

tomas_vid, faltan = {}, []
for t in plano:
    f = comun.TRABAJO / f"toma_{t['n']:02d}.mp4"
    previo = en_carpeta(f.name)
    if previo and t["n"] not in rehacer_v:
        shutil.copy(previo, f); tomas_vid[t["n"]] = f
    elif t["n"] in claves:
        faltan.append(t)
print(f"{len(tomas_vid)} tomas ya hechas (en Drive) · {len(faltan)} por animar · {aw}x{ah}, {pasos} pasos")

def ajustar(img):  # recorta al centro para llenar el cuadro del video sin deformar
    k = max(aw / img.width, ah / img.height)
    img = img.resize((round(img.width * k), round(img.height * k)), Image.LANCZOS)
    x, y = (img.width - aw) // 2, (img.height - ah) // 2
    return img.crop((x, y, x + aw, y + ah))

if faltan:
    # 1) textos → embeddings; el codificador de texto (UMT5) se libera antes de cargar el modelo de video
    cod = WanPipeline.from_pretrained(M5, transformer=None, vae=None, torch_dtype=dtype, device_map="cuda", low_cpu_mem_usage=True)
    emb = {}
    with torch.no_grad():
        for t in faltan:
            emb[t["n"]] = cod.encode_prompt(prompt=t["mueve"], negative_prompt=director.NEGATIVO_VIDEO, do_classifier_free_guidance=True,
                                            device="cuda", dtype=dtype)
    del cod; gc.collect(); torch.cuda.empty_cache()
    # 2) el modelo de video: completo en A100/L4; comprimido (GGUF Q8) en T4
    if GRANDE:
        transformer = WanTransformer3DModel.from_pretrained(M5, subfolder="transformer", torch_dtype=dtype, device_map="cuda",
                                                            low_cpu_mem_usage=True)
    else:
        from diffusers import GGUFQuantizationConfig
        transformer = WanTransformer3DModel.from_single_file(GGUF, quantization_config=GGUFQuantizationConfig(compute_dtype=dtype),
                                                             config=M5, subfolder="transformer", torch_dtype=dtype).to("cuda")
    vae = AutoencoderKLWan.from_pretrained(M5, subfolder="vae", torch_dtype=torch.float32).to("cuda")
    try:
        vae.enable_tiling()
    except Exception:
        pass
    pipe = WanImageToVideoPipeline(tokenizer=None, text_encoder=None, vae=vae, transformer=transformer, expand_timesteps=True,
                                   scheduler=UniPCMultistepScheduler.from_pretrained(M5, subfolder="scheduler"))
    print(f"Modelo de video listo · {torch.cuda.memory_allocated() / 2**30:.1f} GB en la GPU")
    for t in faltan:
        n, dur = t["n"], t["hasta"] - t["desde"]
        pe, ne = emb[n]
        g = torch.Generator("cuda").manual_seed(t["semilla"])
        try:
            with torch.no_grad():
                cuadros = pipe(image=ajustar(Image.open(claves[n]).convert("RGB")), prompt_embeds=pe, negative_prompt_embeds=ne,
                               height=ah, width=aw, num_frames=director.cuadros_para(dur, 24), num_inference_steps=pasos,
                               guidance_scale=5.0, generator=g).frames[0]
        except torch.cuda.OutOfMemoryError:
            gc.collect(); torch.cuda.empty_cache()
            print(f"Toma {n}: no cupo en la GPU; queda como foto con movimiento de cámara."); continue
        arr = np.asarray(cuadros, dtype=np.float32)
        if not np.isfinite(arr).all() or arr.mean() < 0.02:
            print(f"Toma {n}: salió negra (pasa a veces en T4 con fp16). Prueba otra semilla o una GPU L4/A100."); continue
        crudo = comun.TRABAJO / f"toma_{n:02d}_crudo.mp4"
        export_to_video(cuadros, str(crudo), fps=24)
        final = comun.TRABAJO / f"toma_{n:02d}.mp4"
        comun.sh(f'ffmpeg -hide_banner -loglevel error -y -i "{crudo}" -c:v libx264 -crf 18 -preset medium -pix_fmt yuv420p -an "{final}"')
        guardar_en_carpeta(final); tomas_vid[n] = final
        print(f"Toma {n} lista ({t['clase']})"); comun.ver(final, 480)
    del pipe, transformer, vae, emb; gc.collect(); torch.cuda.empty_cache()

In [ ]:
#@title 6 · Labios que cantan la canción (LatentSync)
SINCRONIZAR_LABIOS = True  #@param {type:"boolean"}
PASOS_LABIOS = 20  #@param {type:"slider", min:10, max:50, step:5}
import os
cantan = [t for t in plano if t["canta"] and t["n"] in tomas_vid]
if SINCRONIZAR_LABIOS and cantan:
    if not os.path.exists("/content/ls_env/bin/python"):
        if not os.path.exists("/content/latentsync"):
            comun.sh("git clone -q https://github.com/bytedance/LatentSync /content/latentsync")
            comun.sh("git -C /content/latentsync checkout -q a229c3948406bc2cf6eaf4873e662e70c6a04746")
        comun.sh("pip install -q uv && uv venv -q -p 3.10 /content/ls_env")
        comun.sh("uv pip install -q -p /content/ls_env/bin/python -r requirements.txt", cwd="/content/latentsync")
    LS = "1.6" if VRAM >= 18 else "1.5"
    CONFIG = "configs/unet/stage2_512.yaml" if LS == "1.6" else "configs/unet/stage2.yaml"
    from huggingface_hub import hf_hub_download
    for f in ["whisper/tiny.pt", "latentsync_unet.pt"]:
        hf_hub_download(f"ByteDance/LatentSync-{LS}", f, local_dir="/content/latentsync/checkpoints")
    camino_voz = ((p.get("ia") or {}).get("cancion") or {}).get("voz")
    voz = comun.bajar(camino_voz) if camino_voz else comun.separar_voz(comun.bajar(p["audio"]))[0]
    for t in cantan:
        n = t["n"]
        final = comun.TRABAJO / f"labios_{n:02d}.mp4"
        previo = en_carpeta(final.name)
        if previo:
            shutil.copy(previo, final); tomas_vid[n] = final; continue
        tramo = comun.TRABAJO / f"tramo_{n:02d}.wav"
        comun.sh(f'ffmpeg -hide_banner -loglevel error -y -ss {t["desde"]} -to {t["hasta"]} -i "{voz}" -ac 1 -ar 16000 "{tramo}"')
        crudo = comun.TRABAJO / f"labios_{n:02d}_crudo.mp4"
        try:
            comun.sh(f'/content/ls_env/bin/python -m scripts.inference --unet_config_path "{CONFIG}" '
                     f'--inference_ckpt_path checkpoints/latentsync_unet.pt --inference_steps {PASOS_LABIOS} --guidance_scale 1.5 '
                     f'--enable_deepcache --video_path "{tomas_vid[n]}" --audio_path "{tramo}" --video_out_path "{crudo}"',
                     cwd="/content/latentsync")
        except RuntimeError:
            print(f"Toma {n}: no encontré la cara en toda la toma; queda sin labios sincronizados."); continue
        comun.sh(f'ffmpeg -hide_banner -loglevel error -y -i "{crudo}" -c:v libx264 -crf 18 -pix_fmt yuv420p -an "{final}"')
        guardar_en_carpeta(final); tomas_vid[n] = final
        print(f"Toma {n}: labios listos"); comun.ver(final, 480)
else:
    print("Sin tomas para sincronizar labios.")

In [ ]:
#@title 7 · Montaje final con la canción
FRANJAS_DE_CINE = True  #@param {type:"boolean"}
GRANO = 7  #@param {type:"slider", min:0, max:15, step:1}
cancion = comun.bajar(p["audio"])
clips = {n: str(f) for n, f in tomas_vid.items()}
sin_video = [t["n"] for t in plano if t["n"] not in clips and t["n"] in claves]
for n in sin_video:  # la toma que no se pudo animar entra como foto con un acercamiento lento
    clips[n] = str(claves[n])
if sin_video:
    print("Tomas como foto con movimiento de cámara:", sin_video)
salida = comun.TRABAJO / f"{nombre}_director_{FORMATO}.mp4"
director.montar(plano, clips, cancion, salida, formato=FORMATO, franjas=FRANJAS_DE_CINE, grano=GRANO,
                carpeta=comun.TRABAJO / "montaje")
guardar_en_carpeta(salida)
comun.ver(salida, 720)

In [ ]:
#@title 8 · Guardar en tu repo y poner las tomas en el proyecto
PONER_TOMAS_EN_ESCENAS = True  #@param {type:"boolean"}
#@markdown Cada toma pasa a ser una escena `video` del portal (con su tiempo y su filtro), para ver el karaoke encima y exportar en el portal. Las escenas anteriores quedan guardadas en `escenas_antes_director`.
final = salida
if salida.stat().st_size > 90 * 2**20:  # GitHub no acepta archivos de más de 100 MB
    final = comun.TRABAJO / f"{nombre}_director_{FORMATO}_liviano.mp4"
    comun.sh(f'ffmpeg -hide_banner -loglevel error -y -i "{salida}" -c:v libx264 -crf 26 -preset medium -c:a copy -movflags +faststart "{final}"')
destino = f"medios/videos/{nombre}_director_{FORMATO}.mp4"
comun.guardar(destino, final.read_bytes(), f"Video director: {p.get('titulo')} ({FORMATO})")
FILTROS = {"byn": "byn", "fantasma": "frio", "noche": "frio", "dorado": "calido"}
if PONER_TOMAS_EN_ESCENAS:
    nuevas = []
    for t in plano:
        clip = tomas_vid.get(t["n"])
        if not clip:
            continue
        ruta = f"medios/escenas/{nombre}_toma{t['n']:02d}_{FORMATO}.mp4"
        comun.guardar(ruta, Path(clip).read_bytes(), f"Toma {t['n']}: {t['clase']}")
        e = {"tipo": "video", "desde": t["desde"], "hasta": t["hasta"], "archivo": ruta, "inicioClip": 0,
             "ia": {"director": t["clase"], "prompt": t["mueve"], "labios": "labios_" in Path(clip).name}}
        if t["grado"] in FILTROS:
            e["filtro"] = FILTROS[t["grado"]]
        nuevas.append(e)
    if nuevas:
        p.setdefault("escenas_antes_director", p["escenas"])
        p["escenas"] = nuevas
p["director"] = {"plano": plano, "formato": FORMATO, "video": destino, "video_modelo": "Wan 2.2 TI2V-5B" + ("" if GRANDE else " GGUF Q8"),
                 "imagen_modelo": "RealVisXL", "labios": "LatentSync"}
comun.guardar_proyecto(PROYECTO, p, f"Proyecto: video director para {p.get('titulo')}")
print("Listo. Video en", destino, "· En el portal pulsa «Recargar proyecto».")